In [1]:
import math

def calculate_bearing(lat1, lon1, lat2, lon2):
    """
    Calculate the geodetic bearing from (lat1, lon1) to (lat2, lon2).
    Returns bearing in degrees clockwise from north.
    """
    # convert degrees to radians
    φ1 = math.radians(lat1)
    φ2 = math.radians(lat2)
    Δλ = math.radians(lon2 - lon1)
    
    x = math.sin(Δλ) * math.cos(φ2)
    y = math.cos(φ1) * math.sin(φ2) - math.sin(φ1) * math.cos(φ2) * math.cos(Δλ)
    
    θ = math.atan2(x, y)
    bearing = (math.degrees(θ) + 360) % 360
    return bearing

def calculate_camera_azimuth(lat_cam, lon_cam, lat_pt, lon_pt, pixel_x, width, fovx):
    """
    Calculate the camera's center azimuth (heading) given:
    - camera GPS (lat_cam, lon_cam)
    - target point GPS (lat_pt, lon_pt)
    - pixel coordinates (pixel_x, pixel_y)
    - image dimensions (width, height)
    - horizontal and vertical field of view (fovx, fovy) in degrees
    """
    # 1. Bearing from camera to point
    bearing = calculate_bearing(lat_cam, lon_cam, lat_pt, lon_pt)
    
    # 2. Horizontal pixel offset from image center
    x_offset = pixel_x - (width / 2)
    
    # 3. Convert pixel offset to angular offset
    alpha_rel = (x_offset / width) * fovx
    
    # 4. Compute camera azimuth
    az_cam = (bearing - alpha_rel) % 360
    return az_cam



In [2]:
# Example usage with provided values
lat_cam = 48.26050384702055
lon_cam = 2.706470479996845
lat_pt = 48.267137321427946
lon_pt = 2.6966637429251343

pixel_x = 642
pixel_y = 362
width = 1280
height = 720

fovx = 54.2  # degrees
fovy = 41.7  # degrees

camera_azimuth = calculate_camera_azimuth(
    lat_cam, lon_cam, lat_pt, lon_pt,
    pixel_x, pixel_y, width, height, fovx, fovy
)

print(f"Calculated camera center azimuth: {camera_azimuth:.3f}°")

Calculated camera center azimuth: 315.377°


In [1]:
import math
import numpy as np

def calculate_camera_azimuth_precise(lat_cam, lon_cam,
                                     lat_pt, lon_pt,
                                     pixel_x, width,
                                     fovx_deg):
    # 1. geodetic bearing
    φ1, φ2 = map(math.radians, (lat_cam, lat_pt))
    Δλ = math.radians(lon_pt - lon_cam)
    x = math.sin(Δλ)*math.cos(φ2)
    y = math.cos(φ1)*math.sin(φ2) - math.sin(φ1)*math.cos(φ2)*math.cos(Δλ)
    bearing = (math.degrees(math.atan2(x, y)) + 360) % 360

    # 2. focal length in px
    fovx_rad = math.radians(fovx_deg)
    focal_px = (width/2) / math.tan(fovx_rad/2)

    # 3. angular offset
    dx = pixel_x - width/2
    alpha_rel = math.degrees(math.atan2(dx, focal_px))

    # 4. camera azimuth
    return (bearing - alpha_rel) % 360

# example
az = calculate_camera_azimuth_precise(
    48.26050384702055, 2.706470479996845,
    48.266832809809074, 2.713446193266575,
    pixel_x=314, width=1280, fovx_deg=54.2
)
print(f"{az:.3f}°")  # ≃ 315.38°


50.876°
